In [1]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch version: 2.12.0+cpu
CUDA available: False


In [2]:
import os
import sys

# Add project root to Python path
sys.path.append(os.path.abspath(".."))

In [3]:
import warnings

# Suppress all warnings globally
warnings.filterwarnings("ignore")

In [4]:
# 1. Tokenizer test
from src.features.tokenizer import SimpleTokenizer

tokenizer = SimpleTokenizer()
print(tokenizer.tokenize("I loved this movie"))

['i', 'loved', 'this', 'movie']


In [5]:
# 2. Vocabulary test
from src.features.vocabulary import Vocabulary

vocab = Vocabulary()
tokens = [["i", "love", "movie"], ["i", "hate", "film"]]
vocab.build(tokens)

print(vocab.stoi)
print(vocab.encode(["i", "love", "movie"]))

{'<PAD>': 0, '<UNK>': 1, 'i': 2, 'love': 3, 'movie': 4, 'hate': 5, 'film': 6}
[2, 3, 4]


In [6]:
# 3. Dataset test

from src.data.dataset import SentimentDataset

dataset = SentimentDataset("data/raw/train.csv", tokenizer, vocab)

print(len(dataset))
print(dataset[0])

25000
{'text': tensor([2, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 6, 1,
        1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1,
        1]), 'label': ten

In [7]:
# 4. DataLoader test

from torch.utils.data import DataLoader

from src.features.collate import collate_batch

loader = DataLoader(dataset, batch_size=4, collate_fn=collate_batch)

batch = next(iter(loader))
print(batch["text"].shape)
print(batch["labels"])

torch.Size([4, 289])
tensor([0, 0, 0, 0])


In [8]:
import os
import sys

# Move from notebooks/ → project root
project_root = os.path.abspath("..")
sys.path.append(project_root)

print("PROJECT ROOT SET TO:", project_root)




PROJECT ROOT SET TO: d:\ML-PROJECTS\nlp-sentiment-analysis-ml-engineering-system


In [9]:
from src.utils.config import load_config

config = load_config("../config/config.yaml")

print(config)

{'model': {'type': 'baseline', 'vocab_size': 5000, 'embed_dim': 128, 'num_classes': 2}, 'training': {'lr': 0.001, 'epochs': 1, 'batch_size': 2}, 'paths': {'train_data': 'data/raw/train.csv', 'vocab_path': 'data/raw/vocab.pkl', 'model_output': 'models/model.pt'}, 'mlflow': {'experiment_name': 'sentiment_experiment'}}


In [10]:
from src.models.model_factory import get_model

model = get_model(config)
print(model)

BaselineModel(
  (embedding): Embedding(5000, 128)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)


In [11]:
x = torch.randint(0, 100, (2, 4))
out = model(x)
print(out.shape)

torch.Size([2, 2])


In [12]:
# ============================================================
# MODEL FACTORY TEST CELL
# ------------------------------------------------------------
# Purpose:
# This cell verifies that the model_factory is working correctly.
#
# It checks:
# 1. The factory correctly reads the config
# 2. The correct model (BaselineModel) is instantiated
# 3. The model forward pass works with dummy input
# 4. Output shape matches expected classification format
# ============================================================

import torch

from src.models.model_factory import get_model

# Example config (normally loaded from config.yaml)
config = {
    "model": {
        "type": "baseline",
        "vocab_size": 5000,
        "embed_dim": 128,
        "num_classes": 2
    }
}

# Create model from factory
model = get_model(config)

# Print model architecture
print(model)

# Dummy input: batch_size=2, sequence_length=4
x = torch.randint(0, 5000, (2, 4))

# Forward pass
output = model(x)

# Print output shape
print(output.shape)

BaselineModel(
  (embedding): Embedding(5000, 128)
  (classifier): Linear(in_features=128, out_features=2, bias=True)
)
torch.Size([2, 2])


In [13]:
# ============================================================
# MLflow Tracker TEST CELL
# ------------------------------------------------------------
# Purpose:
# This cell verifies that MLflowTracker initializes correctly
# and can log a simple run, parameters, and metrics.
# ============================================================

from src.tracking.mlflow_tracker import MLflowTracker

tracker = MLflowTracker("sentiment_experiment")

with tracker.start_run(run_name="test_run"):
    tracker.log_params({"lr": 0.001, "batch_size": 2})
    tracker.log_metrics({"accuracy": 0.85, "loss": 0.42})

print("MLflow tracking test completed successfully")

2026/06/09 12:05:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/09 12:05:33 INFO mlflow.store.db.utils: Updating database tables
2026/06/09 12:05:57 INFO mlflow.tracking.fluent: Experiment with name 'sentiment_experiment' does not exist. Creating a new experiment.


MLflow tracking test completed successfully


In [14]:
import mlflow

print("MLflow tracking URI:", mlflow.get_tracking_uri())

MLflow tracking URI: sqlite:///D:/ML-PROJECTS/nlp-sentiment-analysis-ml-engineering-system/notebooks/mlflow.db


In [15]:
import os

print(os.listdir())
print(os.listdir(".."))

['data', 'data_exploration.ipynb', 'mlflow.db', 'sanity_check.ipynb', 'test_torch.ipynb']
['.git', '.vscode', 'api', 'config', 'data', 'logs', 'mlflow.db', 'models', 'notebooks', 'req.txt', 'requirements.txt', 'src', 'tests']


In [16]:
import os

import mlflow

print("TRACKING URI:", mlflow.get_tracking_uri())
print("CURRENT DIR:", os.getcwd())
print("MLRUNS EXISTS:", os.path.exists("mlruns"))
print("NOTEBOOK MLRUNS EXISTS:", os.path.exists("notebooks/mlruns"))

TRACKING URI: sqlite:///D:/ML-PROJECTS/nlp-sentiment-analysis-ml-engineering-system/notebooks/mlflow.db
CURRENT DIR: d:\ML-PROJECTS\nlp-sentiment-analysis-ml-engineering-system\notebooks
MLRUNS EXISTS: False
NOTEBOOK MLRUNS EXISTS: False


In [17]:
# ============================================================
# FIX: FORCE MLflow TO USE PROJECT ROOT STORAGE
# ------------------------------------------------------------
# This ensures all MLflow runs are stored in:
# project_root/mlruns
# instead of notebooks/ directory
# ============================================================

import os

import mlflow

project_root = os.path.abspath("..")

mlflow.set_tracking_uri(f"file:{project_root}/mlruns")

print("NEW TRACKING URI:", mlflow.get_tracking_uri())

NEW TRACKING URI: file:d:\ML-PROJECTS\nlp-sentiment-analysis-ml-engineering-system/mlruns
